# PS-S06E06: CatBoost

This notebook tackles the [**Playground Series – Season 6, Episode 6: Predicting Stellar Class**](https://www.kaggle.com/competitions/playground-series-s6e6), a competition focused on predicting a class label (`GALAXY`, `QSO`, `STAR`) for the `class` target, based on features describing the position, light intensity across SDSS photometric bands, redshift, spectral type, and galaxy population for each observed object.

CatBoost is a strong baseline for tabular problems and often complements XGBoost/LightGBM in ensembles because it handles categorical features natively and robustly — particularly useful here since `spectral_type` and `galaxy_population` are natural categoricals.

In this notebook we:
- Apply astronomy-domain feature engineering per fold (to avoid leakage)
- Train a **CatBoostClassifier** using **Stratified 5-Fold Cross-Validation**
- Optimize for **Balanced Accuracy**, the primary evaluation metric for this competition
- Collect **Out-of-Fold (OOF) probabilities** and test-set predictions for downstream ensembling


## Install Needed Packages

In [1]:
import os
import gc
import sys
import math
import time
import random
import warnings

# Third-party
import numpy as np
import pandas as pd
import catboost as cb
import optuna
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import balanced_accuracy_score
from catboost import CatBoostClassifier, Pool, EFstrType, EFeaturesSelectionAlgorithm, EShapCalcType

import matplotlib.pyplot as plt
import seaborn as sns

from ps_s06e06_feature_engineering import FeatureFactory
from ps_s06e06_model_visualizer import ModelVisualizer
from ps_s06e06_experiment_setup import ExperimentSetup

warnings.filterwarnings('ignore')

import ipywidgets as widgets
widgets.IntProgress()

IntProgress(value=0)

In [2]:
setup = ExperimentSetup(
    model_name='CatBoost',
    use_gpu=True,
    perform_rfe=True,
    perform_optuna_tuning=True
)

# Get the seed and apply it to all internals
seed = setup.set_seeds()

setup.configure_pandas()
setup.suppress_warnings()

# Target label column
TARGET = 'class'

# Class label mapping (must be consistent everywhere)
target_mapping = {'QSO': 0, 'STAR': 1, 'GALAXY': 2}
inverse_target_mapping = {v: k for k, v in target_mapping.items()}

setup.describe()

Random seed set to: 10301
Warnings suppressed.

Experiment Setup
Model         : CatBoost
Target        : class
Use GPU       : True
RFE           : True
Optuna Tuning : True 
On Kaggle     : False


## Read and Examine the Training Dataset

In [3]:
training_df = setup.read_dataset('training')

TRAINING DATASET

   id   alpha  delta      u      g      r      i      z  redshift  \
0   0 147.734 16.959 25.472 21.896 20.358 19.257 18.621     0.409   
1   1 127.989 32.347 20.779 19.087 17.587 17.226 16.786     0.158   
2   2 179.793 35.345 21.035 21.079 21.172 20.583 20.557     2.824   
3   3 225.818 48.569 23.305 21.051 19.018 18.366 17.915     0.536   
4   4 141.836 19.343 21.703 19.472 18.234 17.899 17.616     0.556   

  spectral_type galaxy_population   class  
0             M      Red_Sequence  GALAXY  
1             M      Red_Sequence  GALAXY  
2           O/B        Blue_Cloud     QSO  
3             M      Red_Sequence  GALAXY  
4             M      Red_Sequence  GALAXY  


## Read and Examine the Test Dataset

In [4]:
test_df = setup.read_dataset('test')

TEST DATASET

       id   alpha  delta      u      g      r      i      z  redshift  \
0  577347 120.720 23.924 23.668 21.952 21.086 20.180 19.202     0.429   
1  577348 219.414 42.172 24.903 22.339 20.732 19.860 19.688     0.867   
2  577349 173.569 -1.756 19.428 18.475 17.551 16.571 16.177     0.224   
3  577350 184.904 -1.411 23.121 21.527 20.670 20.418 20.699     0.067   
4  577351 222.488 15.381 25.094 22.644 21.123 19.440 19.094     0.977   

  spectral_type galaxy_population  
0           G/K      Red_Sequence  
1             M      Red_Sequence  
2           G/K        Blue_Cloud  
3           G/K      Red_Sequence  
4             M      Red_Sequence  


## Feature Engineering Strategy

We apply a set of feature engineering steps grounded in SDSS photometric and spectroscopic properties:

- `encoding`: Casts `spectral_type` (ordered O/B→M) and `galaxy_population` (unordered) to CatBoost-native categorical dtype.
- `colors`: Computes SDSS photometric color indices (`u-g`, `g-r`, `r-i`, `i-z`, and wider-baseline combos). These are the primary discriminators in color-color space.
- `ratios`: SED shape features — mean/std magnitude across bands, spectral slope, and outer-to-central band ratio.
- `redshift`: Redshift-derived features (`log1p`, squared, boolean flags) and color×redshift interaction terms. Redshift is the single strongest discriminator in this dataset.
- `position`: Cartesian sky-coordinate projection and |δ|.
- `interactions`: Higher-order cross products between the most discriminating features.

**Categorical handling**: `spectral_type` and `galaxy_population` are passed directly to CatBoost as categoricals. CatBoost learns ordered statistics over them natively without one-hot encoding.

In [5]:
# Define which feature engineering strategies to use
fe_strategies = [
    'encoding',       # Cast spectral_type and galaxy_population to category dtype
    'colors',         # SDSS photometric color indices
    'ratios',         # SED shape features: mean/std magnitude, spectral slope
    'redshift',       # Redshift-derived features and color×redshift interactions
    'position',       # Sky-coordinate Cartesian projection and |delta|
    'interactions',   # Higher-order cross-feature products
    'flux',
    'numeric_expansion'
]

In [6]:
print('Performing initial feature engineering.')

# Initialize and apply the feature factory
feature_engineer = FeatureFactory(strategies=fe_strategies, target=TARGET, seed=seed)

# Encode target labels: QSO=0, STAR=1, GALAXY=2
y = training_df[TARGET].map(target_mapping).astype(int)

# Fit_transform on training, transform on test
X = feature_engineer.fit_transform(training_df)
X_test = feature_engineer.transform(test_df)

cat_features = feature_engineer.get_cat_features(X)
print(f'Feature Count: {X.shape[1]}')
print(f'CatBoost categorical features ({len(cat_features)}): {cat_features}')
print(f'Target sample: {y.head()}')
display(X.head())

Performing initial feature engineering.
Feature Count: 81
CatBoost categorical features (3): ['spectral_type', 'galaxy_population', 'sky_bin']
Target sample: 0    2
1    2
2    0
3    2
4    2
Name: class, dtype: int64


,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,u_minus_g,g_minus_r,r_minus_i,i_minus_z,u_minus_r,u_minus_z,g_minus_z,g_minus_i,color_range,blue_red_slope,color_curvature,mean_mag,std_mag,outer_to_r,redshift_log1p,redshift_sq,is_near_zero_z,is_high_z,z_x_u_g,z_x_g_r,z_x_color_range,is_mid_z,pos_x,pos_y,pos_z,abs_delta,alpha_sin,alpha_cos,delta_sin,delta_cos,alpha_bin,delta_bin,sky_bin,z_x_mean_mag,ug_x_gr,gr_x_ri,std_mag_x_z,color_skew,flux_u,flux_g,flux_r,flux_i,flux_z,flux_u_over_g,total_flux,Log_id,id_sq,id_sqrt,Log_alpha,alpha_sq,alpha_sqrt,Log_delta,delta_sq,delta_sqrt,Log_u,u_sq,u_sqrt,Log_g,g_sq,g_sqrt,Log_r,r_sq,r_sqrt,Log_i,i_sq,i_sqrt,Log_z,z_sq,z_sqrt,Log_redshift,redshift_sqrt
0,147.734,16.959,25.472,21.896,20.358,19.257,18.621,0.409,M,Red_Sequence,3.577,1.538,1.101,0.636,5.114,6.851,3.275,2.638,6.851,2.476,0.437,21.121,2.731,1.083,0.343,0.167,0,0,1.463,0.629,2.802,1,-0.809,0.511,0.292,16.959,0.534,-0.846,0.292,0.957,9,6,9_6,8.638,5.499,1.693,1.117,1.077,0.000,0.000,0.000,0.000,0.000,0.024,0.000,0.000,0,0.000,5.002,"21,825.410",12.155,2.888,287.617,4.118,3.276,648.829,5.047,3.131,479.415,4.679,3.061,414.445,4.512,3.009,370.836,4.388,2.977,346.744,4.315,0.343,0.640
1,127.989,32.347,20.779,19.087,17.587,17.226,16.786,0.158,M,Red_Sequence,1.691,1.500,0.361,0.440,3.191,3.992,2.301,1.861,3.992,1.330,1.139,18.293,1.637,1.068,0.147,0.025,0,0,0.267,0.237,0.631,1,-0.520,0.666,0.535,32.347,0.788,-0.616,0.535,0.845,8,9,8_9,2.890,2.537,0.542,0.259,0.765,0.000,0.000,0.000,0.000,0.000,0.202,0.000,0.693,1,1.000,4.860,"16,381.102",11.313,3.507,"1,046.310",5.687,3.081,431.746,4.558,3.000,364.316,4.369,2.922,309.310,4.194,2.903,296.737,4.150,2.878,281.784,4.097,0.147,0.397
2,179.793,35.345,21.035,21.079,21.172,20.583,20.557,2.824,O/B,Blue_Cloud,-0.044,-0.093,0.589,0.025,-0.137,0.478,0.522,0.496,0.478,-0.633,-0.682,20.885,0.292,0.982,1.341,7.974,0,1,-0.124,-0.262,1.349,0,-0.816,0.003,0.578,35.345,0.004,-1.000,0.578,0.816,11,9,11_9,58.975,0.004,-0.055,0.825,-0.237,0.000,0.000,0.000,0.000,0.000,0.820,0.000,1.099,4,1.414,5.197,"32,325.396",13.409,3.593,"1,249.258",5.945,3.093,442.480,4.586,3.095,444.330,4.591,3.099,448.247,4.601,3.072,423.645,4.537,3.071,422.605,4.534,1.341,1.680
3,225.818,48.569,23.305,21.051,19.018,18.366,17.915,0.536,M,Red_Sequence,2.254,2.033,0.652,0.451,4.287,5.390,3.136,2.685,5.390,1.602,1.381,19.931,2.235,1.084,0.429,0.287,0,0,1.209,1.090,2.890,0,-0.461,-0.475,0.750,48.569,-0.717,-0.697,0.750,0.662,15,12,15_12,10.685,4.583,1.326,1.198,0.807,0.000,0.000,0.000,0.000,0.000,0.099,0.000,1.386,9,1.732,5.424,"50,993.902",15.027,3.903,"2,358.989",6.969,3.191,543.126,4.828,3.093,443.133,4.588,2.997,361.675,4.361,2.964,337.297,4.286,2.940,320.946,4.233,0.429,0.732
4,141.836,19.343,21.703,19.472,18.234,17.899,17.616,0.556,M,Red_Sequence,2.231,1.237,0.335,0.283,3.469,4.087,1.855,1.572,4.087,1.896,0.902,18.985,1.676,1.078,0.442,0.309,0,0,1.240,0.688,2.271,0,-0.742,0.583,0.331,19.343,0.618,-0.786,0.331,0.944,9,6,9_6,10.551,2.761,0.414,0.932,1.162,0.000,0.000,0.000,0.000,0.000,0.121,0.000,1.609,16,2.000,4.962,"20,117.489",11.909,3.013,374.146,4.398,3.123,471.027,4.659,3.019,379.146,4.413,2.957,332.495,4.270,2.939,320.390,4.231,2.924,310.330,4.197,0.442,0.745


In [7]:
# Resolve categorical column list to those actually present after engineering
cat_cols = feature_engineer.get_cat_features(X)
cat_cols = [c for c in cat_cols if c in X.columns]
print(f'Categorical Features: {cat_cols}')

# CatBoost accepts categoricals as strings or integer codes. We use strings.
for c in cat_cols:
    X[c] = X[c].astype(str)
    X_test[c] = X_test[c].astype(str)

Categorical Features: ['spectral_type', 'galaxy_population', 'sky_bin']


### Feature Selection (CatBoost Native)

CatBoost's built-in `select_features` uses SHAP values to iteratively remove the weakest features. This is more appropriate than sklearn RFECV here because it respects the native categorical encoding and uses the same gradient-boosted objective as the final model.

The selection runs on a stratified holdout split (not full CV) for speed, then the selected feature names are applied per-fold during training to prevent leakage.

In [8]:
n_jobs = 1 if setup.use_gpu() else -1

optimal_cols = X.columns

if setup.perform_rfe():
    print('Running CatBoost native feature selection (RecursiveByShapValues)...')

    FS_STEPS = 20
    FS_EVAL_SPLIT = 0.2

    fe_fs = FeatureFactory(strategies=fe_strategies, target=TARGET, seed=seed)
    X_fs = fe_fs.fit_transform(training_df)
    cat_features_fs = fe_fs.get_cat_features(X_fs)

    # Cast cats to string for CatBoost
    for c in cat_features_fs:
        if c in X_fs.columns:
            X_fs[c] = X_fs[c].astype(str)

    sss = StratifiedShuffleSplit(n_splits=1, test_size=FS_EVAL_SPLIT, random_state=seed)
    tr_idx, va_idx = next(sss.split(X_fs, y))

    X_tr_fs, y_tr_fs = X_fs.iloc[tr_idx], y.iloc[tr_idx]
    X_va_fs, y_va_fs = X_fs.iloc[va_idx], y.iloc[va_idx]

    train_pool_fs = Pool(X_tr_fs, y_tr_fs, cat_features=cat_features_fs)
    eval_pool_fs  = Pool(X_va_fs, y_va_fs, cat_features=cat_features_fs)

    fs_model = CatBoostClassifier(
        loss_function='MultiClass',
        eval_metric='Accuracy',
        classes_count=3,
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        feature_border_type='UniformAndQuantiles',
        border_count=254,
        random_seed=seed,
        auto_class_weights='Balanced',
        task_type='GPU' if setup.use_gpu() else 'CPU',
        verbose=200,
        early_stopping_rounds=100,
        allow_writing_files=False,
    )

    fs_result = fs_model.select_features(
        train_pool_fs,
        eval_set=eval_pool_fs,
        features_for_select=list(range(X_tr_fs.shape[1])),
        num_features_to_select=int(X_tr_fs.shape[1] * 0.70),  # keep 70% (tune: 0.6–0.9)
        steps=FS_STEPS,
        algorithm=EFeaturesSelectionAlgorithm.RecursiveByShapValues,
        shap_calc_type=EShapCalcType.Regular,
        train_final_model=False,
        logging_level="Verbose",
        plot=False
    )

    print("\nEliminated Features:", fs_result['eliminated_features_names'])
    print("Selected Features count:", len(fs_result['selected_features_names']))

    loss_graph = fs_result['loss_graph']
    print("Loss values at each step:", loss_graph['loss_values'])

    selected_idx = fs_result["selected_features"]
    optimal_cols = [X_fs.columns[i] for i in selected_idx]
    print(f"\nSelected {len(optimal_cols)} / {X_fs.shape[1]} features.")

else:
    print("Feature selection skipped.  Using features calculated earlier.")
    optimal_cols = pd.Index([
        'alpha',
        'delta',
        'u',
        'g',
        'r',
        'i',
        'z',
        'redshift',
        'u_minus_g',
        'g_minus_r',
        'r_minus_i',
        'i_minus_z',
        'u_minus_r',
        'u_minus_z',
        'g_minus_z',
        'g_minus_i',
        'blue_red_slope',
        'mean_mag',
        'std_mag',
        'outer_to_r',
        'redshift_log1p',
        'redshift_sq',
        'z_x_u_g',
        'z_x_g_r',
        'z_x_color_range',
        'is_mid_z',
        'pos_x',
        'pos_y',
        'pos_z',
        'abs_delta'
    ])
    
print("\nSelected columns:")
print(optimal_cols[:30])

Running CatBoost native feature selection (RecursiveByShapValues)...
Step #1 out of 20
0:	learn: 0.9198268	test: 0.9191161	best: 0.9191161 (0)	total: 34.6ms	remaining: 34.6s
200:	learn: 0.9571531	test: 0.9568910	best: 0.9568910 (200)	total: 3.11s	remaining: 12.4s
400:	learn: 0.9615683	test: 0.9600846	best: 0.9600846 (400)	total: 6.21s	remaining: 9.27s
600:	learn: 0.9632757	test: 0.9613568	best: 0.9613760 (592)	total: 9.22s	remaining: 6.12s
800:	learn: 0.9642670	test: 0.9621781	best: 0.9622159 (795)	total: 12.3s	remaining: 3.06s
999:	learn: 0.9652222	test: 0.9624530	best: 0.9626080 (907)	total: 15.2s	remaining: 0us
bestTest = 0.9626079526
bestIteration = 907
Shrink model to first 908 iterations.
Feature #57 eliminated
Step #2 out of 20
0:	learn: 0.9198268	test: 0.9191161	best: 0.9191161 (0)	total: 23.5ms	remaining: 23.5s
200:	learn: 0.9571531	test: 0.9568910	best: 0.9568910 (200)	total: 3.2s	remaining: 12.7s
400:	learn: 0.9614960	test: 0.9600355	best: 0.9600532 (399)	total: 6.16s	remain

### Optuna: Hyperparameter Tuning

This section tunes CatBoost hyperparameters to maximize **Balanced Accuracy**.

- **Objective**: Arithmetic mean of class-specific recall scores across QSO, STAR, and GALAXY.
- **Weights**: `auto_class_weights='Balanced'` ensures the model does not ignore any stellar class.
- **Validation**: Each trial runs a 5-fold stratified CV loop for robustness.


In [9]:
print("Precomputing fold-wise FE for Optuna (once)...")
skf_cache = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
fold_cache = []

for fold, (train_idx, val_idx) in enumerate(skf_cache.split(training_df, y)):
    X_train_fold = training_df.iloc[train_idx]
    y_train_fold = y.iloc[train_idx]
    X_val_fold   = training_df.iloc[val_idx]
    y_val_fold   = y.iloc[val_idx]

    fe = FeatureFactory(strategies=fe_strategies, target=TARGET, seed=seed)
    X_train_trans = fe.fit_transform(X_train_fold)
    X_val_trans   = fe.transform(X_val_fold)

    if optimal_cols is not None:
        final_cols = [c for c in optimal_cols if c in X_train_trans.columns]
        X_train_trans = X_train_trans[final_cols]
        X_val_trans   = X_val_trans[final_cols]

    cat_features_fold = fe.get_cat_features(X_train_trans)
    if optimal_cols is not None:
        cat_features_fold = [c for c in cat_features_fold if c in final_cols]

    for c in cat_features_fold:
        X_train_trans[c] = X_train_trans[c].astype(str)
        X_val_trans[c]   = X_val_trans[c].astype(str)

    fold_cache.append((
        Pool(X_train_trans, y_train_fold, cat_features=cat_features_fold),
        Pool(X_val_trans,   y_val_fold,   cat_features=cat_features_fold),
        y_val_fold,
        val_idx
    ))

print(f"Cached {len(fold_cache)} folds.")

Precomputing fold-wise FE for Optuna (once)...
Cached 5 folds.


In [10]:
cols_to_keep = list(optimal_cols) if optimal_cols is not None else None

print(f'Training shape: {X.shape}')
print(f'cols_to_keep: {cols_to_keep}')

# Only import if needed.
if setup.use_gpu():
    import torch
    
def objective(trial):
    trial_start = time.perf_counter()
    
    params = {
        'loss_function': 'MultiClass',
        'eval_metric': 'TotalF1:average=Macro',
        'classes_count': 3,
        'auto_class_weights': 'Balanced',
        'iterations': 4000,
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.06),
        'depth': trial.suggest_int('depth', 6, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 3.0, 8.0),
        'one_hot_max_size': 10,
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 200, log=True),
        'leaf_estimation_iterations': trial.suggest_int('leaf_estimation_iterations', 5, 15),
        'bootstrap_type': 'Bernoulli',
        'subsample': trial.suggest_float('subsample', 0.80, 1.0),
        'feature_border_type': 'UniformAndQuantiles',
        'border_count': 254,
        'task_type': 'GPU' if setup.use_gpu() else 'CPU',
        'gpu_ram_part': 0.65,
        'metric_period': 20,
        'allow_writing_files': False,
        'random_seed': seed,
        'verbose': 0,
    }
    
    grow_policy = trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Lossguide'])
    params['grow_policy'] = grow_policy
    if grow_policy == 'Lossguide':
        params['max_leaves'] = trial.suggest_int('max_leaves', 24, 96)
        params['boosting_type'] = 'Plain'

    oof_preds = np.zeros(len(y), dtype=int)

    for fold, (train_pool, val_pool, y_val_fold, val_idx) in enumerate(fold_cache):
        fold_start = time.perf_counter()

        model = CatBoostClassifier(**params)
        model.fit(
            train_pool,
            eval_set=val_pool,
            early_stopping_rounds=300,
            verbose=False
        )

        fold_elapsed = time.perf_counter() - fold_start
        n_trees = model.tree_count_
        ms_per_tree = (fold_elapsed / n_trees) * 1000 if n_trees else float('nan')

        val_preds = model.predict(val_pool).flatten()
        oof_preds[val_idx] = val_preds
        fold_score = balanced_accuracy_score(y_val_fold, val_preds)

        print(f'  Fold {fold+1}: {fold_elapsed:6.1f}s | trees={n_trees:5d} '
              f'| {ms_per_tree:6.2f} ms/tree | bal_acc={fold_score:.5f} '
              f'| grow={grow_policy}, lei={params["leaf_estimation_iterations"]}, '
              f'depth={params["depth"]}')

        del model
        gc.collect()
        
        trial.report(fold_score, step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    
    trial_elapsed = time.perf_counter() - trial_start
    overall = balanced_accuracy_score(y, oof_preds)
    print(f'Trial {trial.number}: {trial_elapsed/60:.1f} min total | OOF bal_acc={overall:.5f}\n')
    return overall

Training shape: (577347, 81)
cols_to_keep: ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'u_minus_g', 'g_minus_r', 'r_minus_i', 'i_minus_z', 'u_minus_r', 'u_minus_z', 'g_minus_z', 'g_minus_i', 'blue_red_slope', 'color_curvature', 'mean_mag', 'std_mag', 'outer_to_r', 'redshift_log1p', 'redshift_sq', 'z_x_u_g', 'z_x_g_r', 'z_x_color_range', 'is_mid_z', 'pos_x', 'pos_y', 'pos_z', 'abs_delta', 'alpha_sin', 'alpha_cos', 'delta_cos', 'sky_bin', 'z_x_mean_mag', 'gr_x_ri', 'std_mag_x_z', 'color_skew', 'flux_g', 'flux_r', 'flux_u_over_g', 'total_flux', 'Log_alpha', 'alpha_sqrt', 'Log_delta', 'delta_sq', 'delta_sqrt', 'u_sqrt', 'g_sqrt', 'Log_r', 'r_sqrt', 'Log_z', 'z_sqrt', 'Log_redshift', 'redshift_sqrt']


In [11]:
# These were the best parameters from an earlier run.  We'll either use them to
# build the model, or seed the new study so trial 0 isn't wasted.
best_params = {
    'learning_rate': 0.052259403904799,
    'depth': 6,
    'l2_leaf_reg': 8.184951619730263,
    'random_strength': 6.7618126841990565,
    'min_data_in_leaf': 24,
    'leaf_estimation_iterations': 13,
    'subsample': 0.9994422948343273,
    'grow_policy': 'SymmetricTree'
}

if setup.perform_optuna_tuning():
    print('Starting CatBoost Optuna Optimization...')

    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)

    study = optuna.create_study(
        study_name='cb_predict_stellar_class_optuna',
        direction='maximize',
        pruner=pruner,
        sampler=optuna.samplers.TPESampler(
            seed=seed,
            n_startup_trials=5,
            multivariate=True,
        )
    )

    # Seed the study with last best parameters.
    study.enqueue_trial(best_params)
    
    study.optimize(
        objective,
        n_trials=25,
        n_jobs=1,
        gc_after_trial=True,
        show_progress_bar=True,
        timeout=18000
    )

    print('Best Score:', study.best_value)
    print('Best Params:', study.best_params)

    best_params = study.best_params
else:
    print('Using placeholder params (run Optuna for tuned values):')
    print(best_params)

[I 2026-06-14 09:49:42,290] A new study created in memory with name: cb_predict_stellar_class_optuna


Starting CatBoost Optuna Optimization...


  0%|          | 0/25 [00:00<?, ?it/s]

  Fold 1:   81.2s | trees= 1819 |  44.64 ms/tree | bal_acc=0.96431 | grow=SymmetricTree, lei=13, depth=6
  Fold 2:   86.9s | trees= 1923 |  45.21 ms/tree | bal_acc=0.96537 | grow=SymmetricTree, lei=13, depth=6
[W 2026-06-14 09:53:34,314] Trial 0 failed with parameters: {'learning_rate': 0.052259403904799, 'depth': 6, 'l2_leaf_reg': 8.184951619730263, 'random_strength': 6.7618126841990565, 'min_data_in_leaf': 24, 'leaf_estimation_iterations': 13, 'subsample': 0.9994422948343273, 'grow_policy': 'SymmetricTree'} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "/home/tarter/anaconda3/envs/kaggle_env/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_814795/3450076899.py", line 50, in objective
    model.fit(
  File "/home/tarter/anaconda3/envs/kaggle_env/lib/python3.12/site-packages/catboost/core.py", line 5547, in fit
   

KeyboardInterrupt: 

### Model Training: Stratified Cross-Validation

Final training uses **5-fold StratifiedKFold**. Within each fold we:

- Fit the feature engineering pipeline on the training split and transform validation/test (leakage prevention)
- Train a `CatBoostClassifier` with early stopping on validation score
- Store **OOF probabilities** and averaged **test probabilities** for ensembling

The overall CV score is Balanced Accuracy on the full OOF vector across all three stellar classes.

In [ ]:
# TARGET MAPPING REFERENCE:
# 0: QSO
# 1: STAR
# 2: GALAXY

metric_period = 20

cb_tuned_params = {
    **best_params,
    'iterations': 16000,
    'loss_function': 'MultiClass',
    'eval_metric': 'TotalF1:average=Macro',
    'classes_count': 3,
    'auto_class_weights': 'Balanced',
    'bootstrap_type': 'Bernoulli',
    'feature_border_type': 'UniformAndQuantiles',
    'border_count': 254,
    'allow_writing_files': False,
    'random_seed': seed,
    'verbose': 200,
    'metric_period': metric_period,
    'task_type': 'GPU' if setup.use_gpu() else 'CPU',
}

print('Final CatBoost CV Training with OOF / Test probabilities')
print('========================================================')
print('Parameters used for training:')
print(cb_tuned_params)

cols_to_keep = list(optimal_cols) if optimal_cols is not None else None

n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

eval_results = []
models = []

oof_probs = np.zeros((len(y), 3), dtype=np.float64)
test_probs = np.zeros((len(test_df), 3), dtype=np.float64)
oof_preds = np.zeros(len(y), dtype=np.int32)

for fold, (train_idx, val_idx) in enumerate(skf.split(training_df, y)):
    print(f'Starting fold {fold+1}/{n_folds}...')

    X_train_fold = training_df.iloc[train_idx]
    y_train_fold = y.iloc[train_idx]
    X_val_fold   = training_df.iloc[val_idx]
    y_val_fold   = y.iloc[val_idx]

    fe_fold = FeatureFactory(strategies=fe_strategies, target=TARGET, seed=seed)
    X_train_trans = fe_fold.fit_transform(X_train_fold)
    X_val_trans   = fe_fold.transform(X_val_fold)
    X_test_trans  = fe_fold.transform(test_df)

    if cols_to_keep is not None:
        final_cols = [c for c in cols_to_keep if c in X_train_trans.columns]
        if not final_cols:
            raise RuntimeError('No overlap between optimal_cols and fold feature columns.')
        X_train_trans = X_train_trans[final_cols]
        X_val_trans   = X_val_trans[final_cols]
        X_test_trans  = X_test_trans[final_cols]

    cat_features_fold = fe_fold.get_cat_features(X_train_trans)
    if cols_to_keep is not None:
        cat_features_fold = [c for c in cat_features_fold if c in final_cols]

    # Cast to string for CatBoost
    for c in cat_features_fold:
        X_train_trans[c] = X_train_trans[c].astype(str)
        X_val_trans[c]   = X_val_trans[c].astype(str)
        X_test_trans[c]  = X_test_trans[c].astype(str)

    train_pool = Pool(X_train_trans, y_train_fold, cat_features=cat_features_fold)
    val_pool   = Pool(X_val_trans,   y_val_fold,   cat_features=cat_features_fold)
    test_pool  = Pool(X_test_trans,               cat_features=cat_features_fold)

    model = cb.CatBoostClassifier(**cb_tuned_params)
    model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=300,
        verbose=200
    )

    eval_results.append(model.get_evals_result())
    models.append(model)

    val_prob = model.predict_proba(val_pool)
    val_pred = model.predict(val_pool).flatten()

    oof_probs[val_idx] = val_prob
    oof_preds[val_idx] = val_pred

    fold_score = balanced_accuracy_score(y_val_fold, val_pred)
    print(f'Fold {fold+1} Balanced Accuracy: {fold_score:.6f}')

    test_probs += model.predict_proba(test_pool) / n_folds

    del model, train_pool, val_pool
    gc.collect()

overall_score = balanced_accuracy_score(y, oof_preds)
print(f'Overall CV Balanced Accuracy: {overall_score:.6f}')

## Visualizing Model Performance

Diagnostic plots to evaluate how well the model distinguishes between the three stellar classes and to detect potential biases or overfitting.

### Learning Curves (Train vs. Validation Log-Loss)
This plot tracks the metric over boosting iterations. A widening gap between train and validation lines indicates overfitting. Early stopping terminates training once the validation score plateaus.

In [ ]:
mviz = ModelVisualizer(model_name='CatBoost')

mviz.plot_learning_curves(eval_results, metric='TotalF1:average=Macro', metric_period=metric_period)

### Feature Importance
Average feature importance across folds.
* **Validation check:** `redshift`, `redshift_sq`, `redshift_log1p`, and `is_high_z` should dominate — redshift is the single strongest discriminator in SDSS stellar classification.
* **color check:** color indices (`g_minus_z`, `u_minus_r`, etc.) should rank in the mid-tier. Raw magnitudes outranking colors suggests the `colors` strategy is not active.
* **Leakage check:** `id` must not appear. `alpha` and `delta` should rank near zero unless the `position` strategy is active.

In [ ]:
mviz.plot_feature_importance(models, show_values=True)

### Distribution Mismatch
Overlays the distribution of true class labels against OOF predicted labels (QSO=0, STAR=1, GALAXY=2). A large divergence on one class suggests the model is systematically misclassifying that stellar type.

In [ ]:
mviz.plot_distribution_mismatch(y_true=y, y_preds=oof_preds)

### ROC Curves (One-vs-Rest)
Each class is treated as the positive class in turn. AUC measures separation from the other two classes. All three curves should sit well above the diagonal; a sagging QSO curve is the most common weak point in stellar classification tasks.

In [ ]:
mviz.plot_multiclass_roc_curve(y_true=y, y_score=oof_probs, title='CatBoost OOF ROC Curve')

### Confusion Matrix (Out-of-Fold)
Shows which stellar classes the model frequently confuses.
* **QSO (0):** QSOs share photometric colors with both galaxies and stars at certain redshifts — off-diagonal entries here are the most common failure mode.
* **STAR (1):** Stars are near-zero redshift and generally the easiest class to separate. High off-diagonal values here warrant investigation.
* **GALAXY (2):** Galaxies span a wide redshift range and color space; moderate confusion with QSOs at high redshift is expected.

In [ ]:
mviz.plot_confusion_matrix(
    y_true=y,
    y_pred=oof_preds,
    classes=['QSO (0)', 'STAR (1)', 'GALAXY (2)'],
    normalize=True,
    title='CatBoost OOF Confusion Matrix (Normalized)'
)

## Prepare Submission

The competition requires the hard predicted class label (`GALAXY`, `QSO`, or `STAR`) for each row in the test set. We derive predictions by taking the argmax of the averaged test probabilities and mapping back to the original string labels.

In [ ]:
submission_df = setup.read_dataset('submission')

numeric_preds = np.argmax(test_probs, axis=1)

# Map numeric predictions back to string class labels
pred_col = [c for c in submission_df.columns if c != 'id'][0]
submission_df[pred_col] = pd.Series(numeric_preds).map(inverse_target_mapping)

print('SAMPLE SUBMISSION')
display(submission_df.head(20))


In [ ]:
submission_df.to_csv('submission.csv', index=False)
print('Saved: submission.csv')

## Save OOF and Test Probabilities

OOF and averaged test-set probabilities are saved for downstream ensembling (blending or stacking).

For this three-class task we save the full probability distribution across `QSO`, `STAR`, and `GALAXY` rather than just the argmax labels. Raw probabilities are essential for:

- **Weighted Blending**: Averaging confidence scores across XGBoost, LightGBM, and CatBoost models.
- **Stacking**: Using these probabilities as meta-features for a secondary model.
- **Calibration Analysis**: Verifying that predicted confidence aligns with actual class frequencies.

In [ ]:
output_dir = 'predictions'
os.makedirs(output_dir, exist_ok=True)

# Column names match the class label mapping: QSO=0, STAR=1, GALAXY=2
prob_cols = ['prob_qso', 'prob_star', 'prob_galaxy']

# OOF probabilities
oof_prob_df = pd.DataFrame(oof_probs, columns=prob_cols)
oof_df = pd.concat([
    pd.DataFrame({'id': training_df['id'].values, 'target': y.values}),
    oof_prob_df
], axis=1)
oof_df.to_csv(f'{output_dir}/catboost_oof_probs.csv', index=False)

# Test probabilities
test_prob_df = pd.concat([
    pd.DataFrame({'id': test_df['id'].values}),
    pd.DataFrame(test_probs, columns=prob_cols)
], axis=1)
test_prob_df.to_csv(f'{output_dir}/catboost_test_probs.csv', index=False)

print(f'Saved for ensembling:\n - {output_dir}/catboost_oof_probs.csv\n - {output_dir}/catboost_test_probs.csv')
